In [1]:
# --- Drift logic (2026+ regime with Overdue & Promotion/Suppression)
from core.paths import DATABASE_EXPORTS_DIR
from datetime import datetime
import pandas as pd
import numpy as np

BASE_PATH = DATABASE_EXPORTS_DIR
PLANNING_YEAR = 2026
PROMOTE_WINDOW_GS = 1
LEGACY_CUTOVER_YEAR = 2026

MILESTONE_MAP_LEGACY = {1: "Survival 1", 3: "Inv 1", 6: "Inv 2", 9: "Inv 3"}
MILESTONE_MAP_NEW = {2: "Survival 1", 4: "Inv 1", 7: "Inv 2", 10: "Inv 3"}
MILESTONE_TYPES = set(["Survival 1", "Inv 1", "Inv 2", "Inv 3"])

# --- Helpers
def milestone_map_for_year(year):
    return MILESTONE_MAP_NEW if year >= LEGACY_CUTOVER_YEAR else MILESTONE_MAP_LEGACY

def compute_gs(planting_year, year):
    """Compute growing season at end of a given planning year."""
    p = pd.to_numeric(planting_year, errors="coerce")
    gs = pd.Series(np.nan, index=p.index)
    ok = p.notna() & (year >= p)
    gs.loc[ok] = (year - p.loc[ok]) + 1
    return gs

def scheduled_type_from_gs(gs, year):
    """Base scheduled type from physiology rules."""
    m = milestone_map_for_year(year)
    out = pd.Series([None] * len(gs), index=gs.index, dtype="object")
    valid = gs.notna() & (gs >= 1) & (gs <= 12)
    is_milestone = valid & gs.isin(list(m.keys()))
    out.loc[is_milestone] = gs.loc[is_milestone].map(m)
    out.loc[valid & ~is_milestone] = "Permanent Plots"
    return out

def next_milestone_for(gs_val, year):
    m = milestone_map_for_year(year)
    if pd.isna(gs_val):
        return None, None
    candidates = sorted([k for k in m if k >= int(gs_val)])
    if not candidates:
        return None, None
    nxt = candidates[0]
    return nxt, m[nxt]

# --- Base: one row per contract
df_base = df_tree_age.copy()
df_base["planting_year_num"] = pd.to_numeric(df_base["planting_year"], errors="coerce")
df_base["gs_at_year_end"] = compute_gs(df_base["planting_year_num"], PLANNING_YEAR)
df_base["scheduled_type_base"] = scheduled_type_from_gs(df_base["gs_at_year_end"], PLANNING_YEAR)

# --- Historical aggregates (for overdue + drift logic)
df_hist = df_meta.copy()
df_hist["event_date"] = pd.to_datetime(df_hist.get("inventory_date"), errors="coerce")
df_hist["event_year"] = np.where(
    df_hist["event_date"].notna(),
    df_hist["event_date"].dt.year,
    pd.to_numeric(df_hist.get("inventory_year"), errors="coerce"),
)
df_hist["planting_year_num"] = pd.to_numeric(df_hist["planting_year"], errors="coerce")
df_hist["gs_at_event"] = np.where(
    df_hist["planting_year_num"].notna() & df_hist["event_year"].notna(),
    (df_hist["event_year"] - df_hist["planting_year_num"]) + 1,
    np.nan,
)
df_hist["executed_type"] = scheduled_type_from_gs(df_hist["gs_at_event"], df_hist["event_year"].fillna(PLANNING_YEAR))

# last inventory per contract
df_hist = df_hist.sort_values("event_date")
last_inv = df_hist.groupby("contract_code", as_index=False).last()[["contract_code", "event_date", "event_year"]]
last_inv = last_inv.rename(columns={"event_date": "last_inventory_date", "event_year": "last_inventory_year"})

# flags for executed milestones
milestone_flags = df_hist.groupby(["contract_code", "executed_type"], as_index=False).size().pivot(
    index="contract_code", columns="executed_type", values="size"
).fillna(0)
for t in MILESTONE_TYPES:
    milestone_flags[f"has_executed__{t}"] = milestone_flags.get(t, 0) > 0
milestone_flags = milestone_flags.reset_index()

# --- Merge
df_plan = df_base.merge(last_inv, on="contract_code", how="left").merge(milestone_flags, on="contract_code", how="left")

# --- Overdue rule
df_plan["years_since_last_inventory"] = np.where(
    df_plan["last_inventory_year"].notna(),
    PLANNING_YEAR - df_plan["last_inventory_year"].astype(float),
    np.nan,
)
if "survival_percent" in df_meta.columns:
    surv_ref = df_meta.groupby("contract_code", as_index=False)["survival_percent"].last()
    df_plan = df_plan.merge(surv_ref, on="contract_code", how="left")
else:
    df_plan["survival_percent"] = np.nan

df_plan["is_overdue"] = (
    df_plan["survival_percent"].astype(float).gt(15)
    & df_plan["years_since_last_inventory"].ge(3)
)
df_plan["overdue_reason"] = np.where(df_plan["is_overdue"], "SURV_GT_15_AND_NO_INV_3Y", None)

# --- Drift promotion / suppression
df_plan["scheduled_type_final"] = df_plan["scheduled_type_base"]
df_plan["schedule_override_flag"] = False
df_plan["schedule_override_reason"] = None

# Promotion (Overdue + base=PP)
is_base_pp = df_plan["scheduled_type_base"].eq("Permanent Plots")
for i, row in df_plan.loc[df_plan["is_overdue"] & is_base_pp].iterrows():
    nxt_gs, nxt_type = next_milestone_for(row["gs_at_year_end"], PLANNING_YEAR)
    if nxt_type and (nxt_gs - row["gs_at_year_end"]) <= PROMOTE_WINDOW_GS:
        executed = bool(row.get(f"has_executed__{nxt_type}", False))
        if not executed:
            df_plan.at[i, "scheduled_type_final"] = nxt_type
            df_plan.at[i, "schedule_override_flag"] = True
            df_plan.at[i, "schedule_override_reason"] = "OVERDUE_PROMOTED_TO_NEXT_MILESTONE_AVOID_CONTIGUOUS"

# Suppression (if already executed)
for i, row in df_plan.iterrows():
    cur_type = row["scheduled_type_final"]
    if cur_type in MILESTONE_TYPES and bool(row.get(f"has_executed__{cur_type}", False)):
        df_plan.at[i, "scheduled_type_final"] = "Permanent Plots"
        df_plan.at[i, "schedule_override_flag"] = True
        df_plan.at[i, "schedule_override_reason"] = "MILESTONE_ALREADY_EXECUTED_SUPPRESSED"

# --- Output
df_plan["rule_version"] = "2026b"
df_plan["computed_at"] = datetime.now().isoformat(timespec="seconds")
OUT_PATH = BASE_PATH / "pt4_inventory_advanced_2026b.xlsx"
df_plan.to_excel(OUT_PATH, index=False)

print("✅ Saved:", OUT_PATH)
display(df_plan.head(25))


NameError: name 'df_tree_age' is not defined